# AgroBot: crop-disease classifier (MobileNetV2), held-out evaluation

Fine-tunes an ImageNet-pretrained **MobileNetV2** on the *Five Crop Diseases* dataset
(17 classes, 5 crops) and evaluates it **once** on a held-out test split.

Leakage controls:
- exact duplicates (MD5) removed; files whose identical copies carry different labels are dropped;
- near-duplicates (same 64-bit difference hash) are kept in the **same split** via a stratified group split (70/15/15);
- model selection uses validation macro-F1 only; the test split is touched once at the end.

In [ ]:
import os, glob, json, hashlib, random, time
from pathlib import Path
from multiprocessing import Pool
import numpy as np, pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = Path('/kaggle/working')
CACHE = Path('/tmp/cache'); CACHE.mkdir(parents=True, exist_ok=True)
NCPU = os.cpu_count()
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '', '| CPUs:', NCPU, '| torch', torch.__version__)
assert DEVICE == 'cuda', 'No GPU: enable a GPU accelerator (requires a phone-verified Kaggle account)'

## 1. Index the dataset

In [ ]:
ROOT = Path(sorted(glob.glob('/kaggle/input/**/Crop___Disease', recursive=True))[0])
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
rows = []
for crop_dir in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    crop = crop_dir.name.capitalize()
    for cls_dir in sorted(p for p in crop_dir.iterdir() if p.is_dir()):
        # sugarcane folders are named without the crop prefix; match the original AgroBot class names
        label = cls_dir.name if '___' in cls_dir.name else f'{crop}_{cls_dir.name}'
        for f in sorted(cls_dir.iterdir()):
            if f.suffix.lower() in EXTS:
                rows.append((str(f), label, crop))
df = pd.DataFrame(rows, columns=['path', 'label', 'crop'])
print(ROOT, '|', len(df), 'images')
print(df.label.value_counts().sort_index().to_string())

## 2. Hash, resize-cache, and de-duplicate

In [ ]:
SHORT = 256

def process(args):
    i, path = args
    with open(path, 'rb') as fh:
        raw = fh.read()
    md5 = hashlib.md5(raw).hexdigest()
    try:
        im = Image.open(path)
        orig = im.size
        im.draft('RGB', (SHORT * 2, SHORT * 2))
        im = im.convert('RGB')
    except Exception as e:
        return i, md5, None, None, None, repr(e)
    g = np.asarray(im.convert('L').resize((9, 8), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    dhash = f"{int(''.join('1' if b else '0' for b in bits), 2):016x}"
    w, h = im.size
    s = SHORT / min(w, h)
    im = im.resize((max(SHORT, round(w * s)), max(SHORT, round(h * s))), Image.BILINEAR)
    out = str(CACHE / f'{i}.jpg')
    im.save(out, quality=95)
    return i, md5, dhash, out, f'{orig[0]}x{orig[1]}', None

t0 = time.time()
with Pool(NCPU) as pool:
    res = pool.map(process, list(enumerate(df.path)), chunksize=32)
res = pd.DataFrame(res, columns=['i', 'md5', 'dhash', 'cache', 'size', 'err']).set_index('i').sort_index()
df = df.join(res)
print(f'processed in {time.time() - t0:.0f}s')

stats = {'n_raw': len(df), 'n_unreadable': int(df.err.notna().sum())}
df = df[df.err.isna()].copy()

# exact duplicates: drop label conflicts entirely, otherwise keep one copy
labels_per_md5 = df.groupby('md5').label.nunique()
conflict = labels_per_md5[labels_per_md5 > 1].index
stats['n_label_conflict_files'] = int(df.md5.isin(conflict).sum())
df = df[~df.md5.isin(conflict)]
before = len(df)
df = df.drop_duplicates('md5').copy()
stats['n_exact_duplicates_removed'] = before - len(df)

# near-duplicate groups (identical dHash) stay together in one split
stats['n_images_used'] = len(df)
stats['n_dhash_groups'] = int(df.dhash.nunique())
stats['n_images_in_multi_image_groups'] = int(df.dhash.map(df.dhash.value_counts()).gt(1).sum())
print(json.dumps(stats, indent=1))

## 3. Stratified group split (70 / 15 / 15)

In [ ]:
CLASSES = sorted(df.label.unique())
NC = len(CLASSES)
df['y'] = df.label.map({c: k for k, c in enumerate(CLASSES)})

# 20 folds of ~5%: folds 0-2 -> test, 3-5 -> val, rest -> train
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
fold = np.empty(len(df), dtype=int)
for k, (_, idx) in enumerate(sgkf.split(df, df.y, groups=df.dhash)):
    fold[idx] = k
df['split'] = np.where(fold < 3, 'test', np.where(fold < 6, 'val', 'train'))

for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    shared = set(df[df.split == a].dhash) & set(df[df.split == b].dhash)
    assert not shared, f'group leak {a}/{b}'
print(pd.crosstab(df.label, df.split)[['train', 'val', 'test']].to_string())
print(df.split.value_counts().to_string())
df[['path', 'label', 'split', 'md5', 'dhash']].to_csv(OUT / 'split.csv', index=False)

## 4. Data loaders

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, d, tf):
        self.p, self.y, self.tf = d.cache.values, d.y.values, tf
    def __len__(self):
        return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]

BS = 64
mk = lambda s, tf, sh: DataLoader(ImgDS(df[df.split == s], tf), batch_size=BS, shuffle=sh,
                                  num_workers=NCPU, pin_memory=True, persistent_workers=True)
train_dl, val_dl, test_dl = mk('train', train_tf, True), mk('val', eval_tf, False), mk('test', eval_tf, False)

## 5. Fine-tune MobileNetV2

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.last_channel, NC)
model = model.to(DEVICE)

EPOCHS, LR = 12, 5e-4
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
crit = nn.CrossEntropyLoss()

@torch.no_grad()
def predict(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == 'cuda'):
            P.append(model(x.to(DEVICE, non_blocking=True)).float().softmax(1).cpu())
        Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

history, best_f1, best_ep = [], -1, -1
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); tl, tc, tn = 0.0, 0, 0
    for x, y in train_dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == 'cuda'):
            out = model(x); loss = crit(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
    P, Y = predict(val_dl); pred = P.argmax(1)
    rec = dict(epoch=ep, train_loss=tl / tn, train_acc=tc / tn,
               val_acc=accuracy_score(Y, pred), val_macro_f1=f1_score(Y, pred, average='macro'),
               sec=time.time() - t0)
    history.append(rec)
    if rec['val_macro_f1'] > best_f1:
        best_f1, best_ep = rec['val_macro_f1'], ep
        torch.save({'model_state_dict': model.state_dict(), 'class_names': CLASSES, 'epoch': ep}, OUT / 'best_model.pth')
    print(' | '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in rec.items()))
print('best epoch', best_ep, 'val macro-F1', round(best_f1, 4))

## 6. Held-out test evaluation (run once)

In [ ]:
model.load_state_dict(torch.load(OUT / 'best_model.pth', map_location=DEVICE, weights_only=True)['model_state_dict'])
P, Y = predict(test_dl); pred = P.argmax(1)

rng = np.random.default_rng(SEED); boot_acc, boot_f1 = [], []
for _ in range(1000):
    s = rng.integers(0, len(Y), len(Y))
    boot_acc.append(accuracy_score(Y[s], pred[s])); boot_f1.append(f1_score(Y[s], pred[s], average='macro'))
ci = lambda v: [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]

crop_of = np.array([c.split('_')[0] for c in CLASSES])
report = classification_report(Y, pred, target_names=CLASSES, digits=4, output_dict=True)
test = dict(
    n=int(len(Y)),
    accuracy=round(float(accuracy_score(Y, pred)), 4), accuracy_95ci=ci(boot_acc),
    macro_f1=round(float(f1_score(Y, pred, average='macro')), 4), macro_f1_95ci=ci(boot_f1),
    weighted_f1=round(float(f1_score(Y, pred, average='weighted')), 4),
    top3_accuracy=round(float(np.mean([y in np.argsort(p)[-3:] for p, y in zip(P, Y)])), 4),
    crop_level_accuracy=round(float(np.mean(crop_of[pred] == crop_of[Y])), 4),
    per_class_f1={c: round(report[c]['f1-score'], 4) for c in CLASSES},
    per_class_support={c: int(report[c]['support']) for c in CLASSES})
print(classification_report(Y, pred, target_names=CLASSES, digits=4))
print(json.dumps({k: v for k, v in test.items() if not k.startswith('per_class')}, indent=1))

cm = confusion_matrix(Y, pred, normalize='true')
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
short = [c.replace('___', ': ').replace('_', ' ') for c in CLASSES]
ax.set_xticks(range(NC)); ax.set_xticklabels(short, rotation=75, ha='right', fontsize=8)
ax.set_yticks(range(NC)); ax.set_yticklabels(short, fontsize=8)
for a in range(NC):
    for b in range(NC):
        if cm[a, b] >= 0.01:
            ax.text(b, a, f'{cm[a, b]:.2f}', ha='center', va='center', fontsize=6.5,
                    color='white' if cm[a, b] > 0.5 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f"MobileNetV2 test set (n={test['n']}): acc {test['accuracy']:.3f}, macro-F1 {test['macro_f1']:.3f}")
fig.colorbar(im, fraction=0.046); fig.tight_layout(); fig.savefig(OUT / 'confusion_matrix.png', dpi=150); plt.show()

## 7. Model size and inference speed

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
size_mb = os.path.getsize(OUT / 'best_model.pth') / 1e6
x1 = torch.randn(1, 3, 224, 224)

cpu_model = models.mobilenet_v2(num_classes=NC)
cpu_model.load_state_dict({k: v.cpu() for k, v in model.state_dict().items()}); cpu_model.eval()
with torch.inference_mode():
    for _ in range(20): cpu_model(x1)
    ts = []
    for _ in range(100):
        t0 = time.perf_counter(); cpu_model(x1); ts.append(time.perf_counter() - t0)
cpu_ms = float(np.median(ts) * 1e3)

gpu_ips = None
if DEVICE == 'cuda':
    xb = torch.randn(64, 3, 224, 224, device=DEVICE); model.eval()
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for _ in range(10): model(xb)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(50): model(xb)
        torch.cuda.synchronize()
    gpu_ips = float(50 * 64 / (time.perf_counter() - t0))

speed = dict(params_millions=round(n_params / 1e6, 2), checkpoint_mb=round(size_mb, 1),
             cpu_latency_ms_batch1=round(cpu_ms, 1), cpu_threads=torch.get_num_threads(),
             gpu=torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None,
             gpu_images_per_sec_fp16_bs64=round(gpu_ips) if gpu_ips else None)
print(json.dumps(speed, indent=1))

## 8. Save metrics

In [ ]:
metrics = dict(dataset='shubham2703/five-crop-diseases-dataset', classes=CLASSES, seed=SEED,
               data=stats, split_sizes=df.split.value_counts().to_dict(),
               training=dict(arch='mobilenet_v2 (ImageNet-pretrained, full fine-tune)', epochs=EPOCHS, lr=LR,
                             optimizer='AdamW + OneCycle', batch_size=BS, best_epoch=best_ep,
                             best_val_macro_f1=round(best_f1, 4), history=history),
               test=test, speed=speed)
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=1))
print('saved metrics.json, confusion_matrix.png, split.csv, best_model.pth')